# Company Investment Suggestion Model (Multi‑Output)
XGBoost + Pipelines


## 1. Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import joblib

df = pd.read_csv("company_dataset_v2_realistic.csv")

y1 = df["investment_ready"]
y2 = df["recommended_investor_type"]
y3 = df["suggested_funding_range"]

le_investor = LabelEncoder()
le_funding = LabelEncoder()

y2_enc = le_investor.fit_transform(y2)
y3_enc = le_funding.fit_transform(y3)

X = df.drop(["investment_ready", "recommended_investor_type", "suggested_funding_range"], axis=1)

categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

categorical_cols, numerical_cols

(Index(['sector', 'business_model'], dtype='object'),
 Index(['company_age_years', 'annual_revenue_lakhs', 'profit_margin_pct',
        'monthly_burn_rate_lakhs', 'market_size_cr', 'team_size',
        'required_investment_lakhs', 'monthly_active_users', 'growth_rate_pct'],
       dtype='object'))

## 2. Train–Test Split

In [2]:
X_train, X_test, y1_train, y1_test, y2_train, y2_test, y3_train, y3_test = train_test_split(
    X, y1, y2_enc, y3_enc, test_size=0.2, random_state=42
)

## 3. Preprocessing + Pipelines

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

pipeline_ready = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=300,
        random_state=42,
        use_label_encoder=False
    ))
])

pipeline_investor = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(le_investor.classes_),
        eval_metric='mlogloss',
        n_estimators=300,
        random_state=42,
        use_label_encoder=False
    ))
])

pipeline_funding = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(le_funding.classes_),
        eval_metric='mlogloss',
        n_estimators=300,
        random_state=42,
        use_label_encoder=False
    ))
])

## 4. Train Models

In [4]:
pipeline_ready.fit(X_train, y1_train)
pipeline_investor.fit(X_train, y2_train)
pipeline_funding.fit(X_train, y3_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:35:05] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:35:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:35:09] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  Index(['company_age_years', 'annual_revenue_lakhs', 'profit_margin_pct',
       'monthly_burn_rate_lakhs', 'market_size_cr', 'team_size',
       'required_investment_lakhs', 'monthly_active_users', 'growth_rate_pct'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handl...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None, num_class=3, ...))])

## 5. Evaluation

In [5]:
pred_ready = pipeline_ready.predict(X_test)
pred_investor = pipeline_investor.predict(X_test)
pred_funding = pipeline_funding.predict(X_test)

print("Accuracy Ready:", accuracy_score(y1_test, pred_ready))
print("Accuracy Investor:", accuracy_score(y2_test, pred_investor))
print("Accuracy Funding:", accuracy_score(y3_test, pred_funding))

print(classification_report(y1_test, pred_ready))
print(classification_report(y2_test, pred_investor, target_names=le_investor.classes_))
print(classification_report(y3_test, pred_funding, target_names=le_funding.classes_))

Accuracy Ready: 0.827
Accuracy Investor: 0.992
Accuracy Funding: 0.99
              precision    recall  f1-score   support

           0       0.62      0.57      0.59       221
           1       0.88      0.90      0.89       779

    accuracy                           0.83      1000
   macro avg       0.75      0.73      0.74      1000
weighted avg       0.82      0.83      0.82      1000

              precision    recall  f1-score   support

       Angel       0.99      1.00      1.00       974
   Seed Fund       0.87      0.80      0.83        25
          VC       0.00      0.00      0.00         1

    accuracy                           0.99      1000
   macro avg       0.62      0.60      0.61      1000
weighted avg       0.99      0.99      0.99      1000

              precision    recall  f1-score   support

       ₹0–3L       0.99      1.00      1.00       973
     ₹15–60L       0.00      0.00      0.00         1
      ₹3–15L       0.83      0.77      0.80        26

    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

## 6. Save Models

In [6]:
joblib.dump(pipeline_ready, "pipeline_investment_ready.pkl")
joblib.dump(pipeline_investor, "pipeline_investor_type.pkl")
joblib.dump(pipeline_funding, "pipeline_funding_range.pkl")

joblib.dump(le_investor, "label_encoder_investor.pkl")
joblib.dump(le_funding, "label_encoder_funding.pkl")

['label_encoder_funding.pkl']